# Heart Disease Prediction — EDA & Model Comparison

**Problem:** Predict whether a patient has heart disease (binary classification) from 13 clinical measurements.

**Dataset:** UCI Heart Disease (Cleveland subset) — 303 records, 13 features.
Source: https://archive.ics.uci.edu/dataset/45/heart+disease
Citation: Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989). *Heart Disease* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X

**Target:** `target = 0` → no disease, `target = 1` → disease present (original 1–4 diagnoses collapsed to 1).

**This notebook covers:** data understanding → EDA → preprocessing → model comparison (6 algorithms) → hyperparameter tuning → final evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. Load and Understand the Data

In [ ]:
from pathlib import Path

# Works whether the kernel starts in the project root or in notebooks/
for p in ["../data/dataset.csv", "data/dataset.csv"]:
    if Path(p).exists():
        DATA_PATH = p
        MODEL_PATH = str(Path(p).parent.parent / "model.pkl")
        break

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
# Data types and basic stats
df.info()
df.describe()

In [ ]:
# Missing values and duplicate rows
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Class distribution of the target
print(df["target"].value_counts())
sns.countplot(x="target", data=df)
plt.title("Target class distribution (0 = no disease, 1 = disease)")
plt.show()

**Observation:** The classes are fairly balanced (164 healthy / 139 disease), so accuracy is a usable metric — but we still report precision, recall and F1.

## 2. Exploratory Data Analysis (EDA)

### 2.1 Numeric feature distributions

In [ ]:
numeric_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), numeric_features):
    sns.histplot(data=df, x=col, hue="target", kde=True, ax=ax)
    ax.set_title(f"{col} by target")
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

### 2.2 Categorical features vs target

In [ ]:
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), categorical_features):
    sns.countplot(data=df, x=col, hue="target", ax=ax)
    ax.set_title(f"{col} by target")
plt.tight_layout()
plt.show()

### 2.3 Correlation heatmap

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation matrix")
plt.show()

### 2.4 Key EDA insights

Based on the charts above:

1. **`cp` (chest pain type)** — asymptomatic patients (cp=4) show the highest disease rate.
2. **`thalach` (max heart rate)** — patients with disease tend to reach *higher* max heart rates.
3. **`oldpeak` (ST depression)** — higher values are linked to disease.
4. **`exang` (exercise-induced angina)** — presence of angina strongly links to disease.
5. **`ca` (number of colored vessels)** — more vessels → higher disease rate.
6. **`age`** — disease appears across all ages, weak linear correlation alone.
7. **`chol`** — wide spread; not a strong single predictor in this dataset.
8. Missing values exist only in `ca` (4) and `thal` (2) — handled inside the pipeline with imputation, so it also works at prediction time.

## 3. Preprocessing & Train-Test Split

- **Numeric features** (`age, trestbps, chol, thalach, oldpeak`): median imputation + standard scaling.
- **Categorical features** (`sex, cp, fbs, restecg, exang, slope, ca, thal`): most-frequent imputation + one-hot encoding.
- Everything lives inside a `ColumnTransformer` that is part of the model `Pipeline` — this prevents data leakage and makes preprocessing reproducible at prediction time.
- Data is split **before** any fitting: 80% train / 20% test, stratified, `random_state=42` for reproducibility.

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

## 4. Model Comparison (5-fold Cross-Validation)

We compare **six** suitable classification algorithms on the training set using 5-fold CV accuracy.
Cross-validation trains the model on 5 different train/validation splits, so the score is more reliable than a single split.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "K-Nearest Neighbours": KNeighborsClassifier(),
    "Support Vector Machine": SVC(probability=True, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "Naive Bayes": GaussianNB(),
}

results = []
for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    results.append({"Model": name, "CV Accuracy (mean)": scores.mean(), "CV Accuracy (std)": scores.std()})
    print(f"{name:<25} {scores.mean():.4f} +/- {scores.std():.4f}")

results_df = pd.DataFrame(results).sort_values("CV Accuracy (mean)", ascending=False)
results_df

In [ ]:
sns.barplot(data=results_df, x="CV Accuracy (mean)", y="Model")
plt.xlim(0.5, 1.0)
plt.title("5-Fold CV Accuracy comparison")
plt.show()

## 5. Hyperparameter Tuning (GridSearchCV)

The two strongest models (Logistic Regression and Random Forest) are tuned with GridSearchCV (cv=5).
GridSearchCV tries every combination of the given hyperparameter grid and keeps the one with the best cross-validation score.

In [ ]:
param_grids = {
    "Logistic Regression": {
        "classifier__C": [0.01, 0.1, 1, 10, 100],
    },
    "Random Forest": {
        "classifier__n_estimators": [50, 100, 200],
        "classifier__max_depth": [None, 5, 10, 20],
        "classifier__min_samples_split": [2, 5, 10],
    },
}

best_estimators = {}
for name, grid in param_grids.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", models[name])])
    gs = GridSearchCV(pipe, grid, cv=5, scoring="accuracy", n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    best_estimators[name] = gs
    print(f"{name}: best CV accuracy = {gs.best_score_:.4f}, params = {gs.best_params_}")

In [ ]:
best_name = max(best_estimators, key=lambda k: best_estimators[k].best_score_)
best_pipeline = best_estimators[best_name].best_estimator_
print("Selected final model:", best_name)

## 6. Final Evaluation on the Held-Out Test Set

The test set is touched **only now** — after all model selection and tuning — so it gives an unbiased estimate of performance on unseen data.

In [ ]:
y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))

In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Disease", "Disease"], yticklabels=["No Disease", "Disease"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (test set)")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, label=f"{best_name} (AUC = {roc_auc_score(y_test, y_prob):.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (test set)")
plt.legend()
plt.show()

**How to read the metrics:**

- **Accuracy** — overall fraction of correct predictions.
- **Precision** — of those predicted sick, how many really are sick (avoids false alarms).
- **Recall** — of those really sick, how many were caught (avoids missed cases). In medical screening, high recall matters most.
- **F1-score** — harmonic mean of precision and recall.
- **ROC-AUC** — how well the model ranks sick above healthy; 0.5 = random, 1.0 = perfect.

We did not pick the final model because one metric was marginally higher — Logistic Regression won on CV accuracy, is simple, fast, interpretable (coefficients), and performed strongly on recall/ROC-AUC, which fits a medical screening problem on a small dataset.

## 7. Save the Final Fitted Pipeline

The complete fitted pipeline (preprocessing + model) is saved with pickle.
`app.py` loads this exact object for predictions — training is never repeated in the app.

In [ ]:
import pickle

with open(MODEL_PATH, "wb") as f:
    pickle.dump(best_pipeline, f)
print("Saved", MODEL_PATH)